# 第20章 ICLR 2025 Oral | 单卡3090纯视觉玩MineCraft——LS-Imagine

> "不靠API、不看坐标、不要特权信息——让AI像人类一样，只用'眼睛'（纯视觉）来探索Minecraft的开放世界。"

## 1. 知识地图
```
20.1 简介：开放世界决策的三大特征
    ├── 广阔状态空间 + 高度灵活策略 + 环境感知不确定性
    ├── Voyager(API) / DECKARD(无模型RL) / DreamerV3(有模型RL) 的局限
    └── 核心挑战：如何提高庞大状态空间中的探索效率？
20.2 核心创新：四个贡献
    ├── 长短期结合的世界模型架构
    ├── 通过图像放大生成"功用性图"
    ├── 功用性驱动的内在奖励机制
    └── 结合长期价值估计的行为学习
20.3 方法详解
    ├── 功用性图：滑动窗口+MineCLIP虚拟探索
    ├── 快速功也图：Swin-UNet多模态生成
    ├── 跳跃式状态转换：从s_t直接跳到s_{t+H}
    ├── 长短期世界模型：短期+长期两个分支
    └── Bootstrap λ-returns：混合长短期价值估计
20.4 实验结果
    ├── 5个Minecraft任务 (砍树/取水/采沙/剪羊毛/采矿)
    ├── LS-Imagine vs VPT/STEVE-1/PTGM/Director/DreamerV3
    └── 所有任务上显著最优
20.5 总结
```

## 2. 研究背景与动机

### 2.1 开放世界决策的三大挑战

Minecraft 是典型的开放世界游戏，具有三大特征：

1. **广阔的状态空间**：3D 世界中智能体可以在任何位置、面对任何方向
2. **高度灵活的策略**：需要与各种物体交互（砍树、挖掘、取水、剪羊毛...）
3. **环境感知的不确定性**：只能看到第一人称视角的图像，无法直接获知内部状态

**核心挑战：** 在如此庞大的状态空间中，如何高效探索以完成稀疏奖励的长期任务？

### 2.2 现有方法的局限

| 方法 | 类型 | 问题 |
|------|------|------|
| **Voyager** | 高层API控制 | 依赖特定环境的API，不是纯视觉控制 |
| **DECKARD** | 无模型RL | 缺乏对环境机制的理解，样本效率低 |
| **DreamerV3** | 有模型RL | 仅依赖短期经验优化策略 → **"短视"问题** |

**"短视"问题：** DreamerV3 等世界模型在想象时只能模拟连续的短期未来（如5步）。这就像一个人只能看到脚下10米的路，无法规划去远方目标的路径。

### 2.3 LS-Imagine 的核心创新

让世界模型能高效模拟特定行为的长期影响，无需反复逐步预测。

**直觉类比：** 不必一步步模拟从家走到超市的每一步，可以直接"跳跃"想象"我在超市门口"这个状态，然后评估其价值。

## 3. 功用性图 (Affordance Map)——AI的探索地图

### 3.1 什么是功用性图？

功用性图 = 一张与观测图像同大小的热力图，标注了图像中每个区域对完成任务的潜在价值。

- 高亮区域 → 该区域可能与任务目标相关（如可能有树木）
- 暗淡区域 → 不太可能与任务相关

### 3.2 功用性图的计算（虚拟探索法）

1. 在单帧观测图像上，滑动边界框从左到右、从上到下扫描
2. 对每个边界框位置：
   - 裁剪出该区域 + 生成16帧连续放大图像（模拟智能体向该区域移动）
   - 用 **MineCLIP**（预训练的视频-文本对齐模型）评估这段"虚拟探索视频"与任务描述的相关性
3. 融合所有位置的相关性值 → 完整的功用性图

**关键创新：不依赖真实成功轨迹！**

### 3.3 快速功用性图生成（Swin-UNet）

滑动窗口方法太慢。用 Swin-UNet 学习：
- 输入：视觉观察 + 语言指令
- 输出：功用性图
- 训练：用慢方法生成的功也图作为监督信号

In [ ]:
print("=" * 60)
print("功用性图 (Affordance Map) 的概念")
print("=" * 60)
print()
print("计算流程（慢方法——用于生成训练数据）：")
print("  1. 滑动边界框扫描整张观察图像")
print("  2. 对每个边界框位置：")
print("     a. 裁剪 → 生成16帧连续放大图像")
print("     b. 用 MineCLIP 评估这段'虚拟探索视频'")
print("        与任务描述('砍一棵树')的相关性")
print("  3. 融合所有位置的相关性 → 功用性图")
print()
print("快速方法（Swin-UNet——用于实时推理）：")
print("  输入: (观测图像, 语言指令)")
print("  输出: 功用性图")
print()
print("关键发现：即使目标被遮挡/不可见，功用性图仍有效！")
print("  寻找村庄：村庄不在视野 → 建议向开阔区域探索")
print("  挖矿：矿石在地下 → 指示向山体/地面挖掘")

## 4. 跳跃式状态转换

### 4.1 核心矛盾
如果没有真实数据表示智能体已达成目标，如何训练模型从当前状态跳跃到"与目标相关的未来状态"？

### 4.2 跳跃标志
什么时候跳跃 vs 单步？

通过功用性图的**峰度(Kurtosis)**判断：
- 高价值区域集中 → 峰度高 → 远处有目标 → **跳跃**
- 高价值区域分散 → 峰度低 → 附近就有 → **单步**

### 4.3 长短期世界模型架构

```
输入: 状态 s_t + 功用性图 M_t + 动作 a_t
    │
    ├─→ 跳跃预测器 → 选短期还是长期分支
    ├─→ 短期分支 → 预测 s_{t+1} (单步)
    └─→ 长期分支 → 预测 s_{t+H} (跳跃)
          ├─→ 间隔预测器: 估计 Δ_t (跳了多少步)
          └─→ 累积奖励预测: G_t (期间总奖励)
```

In [ ]:
print("=" * 60)
print("长短期世界模型架构 (伪代码)")
print("=" * 60)
print()
print("class LongShortTermWorldModel:")
print("    def forward(self, state, affordance_map, action):")
print("        z = encoder(state, affordance_map)")
print("        jump_prob = jump_predictor(z)")
print()
print("        if jump_prob < threshold:  # 短期")
print("            z_next = short_term_branch(z, action)")
print("            delta_t, G_t = 1, reward(z, action)")
print("        else:  # 长期(跳跃)")
print("            z_next = long_term_branch(z, action)")
print("            delta_t = interval_predictor(z, action)")
print("            G_t = cumulative_reward_predictor(z, action)")
print()
print("        return z_next, delta_t, G_t")

## 5. 行为学习：Bootstrap λ-Returns

在长短期想象序列上计算每个状态的折扣累积奖励：

$$R_t^\lambda = 
\begin{cases}
\hat{c}_t \left[\hat{G}_{t+1} + \gamma^{\hat{\Delta}_{t+1}} ((1-\lambda) v_\psi(\hat{s}_{t+1}) + \lambda R_{t+1}^\lambda)\right] & t < L \\
v_\psi(\hat{s}_L) & t = L
\end{cases}$$

符号：
- $\hat{c}_t$：继续标志；$\hat{G}_{t+1}$：跳跃期间的累积奖励
- $\hat{\Delta}_{t+1}$：跳跃步数间隔；$\gamma$：折扣因子
- $\lambda$：TD(λ) 参数；$v_\psi$：Critic 价值函数
- $L$：想象范围

**关键：** 当用跳跃式转换时，$\hat{\Delta}_{t+1} > 1$，直接注入远期回报！

## 6. 实验结果

### 五个 Minecraft 任务

| 任务 | 语言描述 | 最大步数 |
|------|------|------|
| 砍树 | Cut a tree. | 1000 |
| 取水 | Obtain water. | 1000 |
| 采沙 | Obtain sand. | 1000 |
| 剪羊毛 | Obtain wool. | 1000 |
| 采矿 | Mine iron ore. | 2000 |

### 成功率对比

| 方法 | 砍树 | 取水 | 采沙 | 剪羊毛 | 采矿 |
|------|------|------|------|------|------|
| VPT | 6.97% | 0.61% | 12.99% | 1.94% | 0.00% |
| STEVE-1 | 57.00% | 6.00% | 37.00% | 3.00% | 0.00% |
| DreamerV3 | 53.33% | 55.72% | 59.88% | 25.13% | 16.79% |
| **LS-Imagine** | **80.63%** | **77.31%** | **62.68%** | **54.28%** | **20.28%** |

LS-Imagine 在所有任务上显著最优，尤其在稀疏奖励场景优势更明显。

In [ ]:
# 功用性驱动的内在奖励计算
import torch

print("=" * 60)
print("功用性驱动的内在奖励")
print("=" * 60)
print()

def affordance_reward(affordance_map, sigma=0.3):
    """二维高斯 × 功用性图 → 内在奖励"""
    H, W = affordance_map.shape
    y, x = torch.meshgrid(
        torch.linspace(-1, 1, H),
        torch.linspace(-1, 1, W), indexing='ij')
    gaussian = torch.exp(-(x**2 + y**2) / (2*sigma**2))
    return (affordance_map * gaussian).mean()

print("场景1：目标在视野中心")
m1 = torch.zeros(64,64); m1[20:44,20:44] = 1.0
print(f"  奖励: {affordance_reward(m1).item():.4f} (高！)")

print("场景2：目标在视野边缘")
m2 = torch.zeros(64,64); m2[4:28,4:28] = 1.0
print(f"  奖励: {affordance_reward(m2).item():.4f} (低)")

print("场景3：没有明显目标")
m3 = torch.ones(64,64) * 0.1
print(f"  奖励: {affordance_reward(m3).item():.4f} (很低)")
print()
print("作用：引导智能体将目标对齐到视野中心")

## 7. 常见误区与易错点

### 误区1：LS-Imagine = Dreamer的简单改进
**纠正：** 功用性图 + 跳跃式状态转换 + 混合想象训练是全新范式。

### 误区2：功用性图 = 目标检测
**纠正：** 目标被遮挡/不可见时功用性图仍有效（依赖MineCLIP学到的上下文知识）。

### 误区3：跳跃式转换 = 跳帧
**纠正：** 跳帧是丢弃中间帧；跳跃通过**学习**预测跳跃后的状态和累积奖励。

### 误区4：单卡3090就能复现意味着很简单
**纠正：** MineCLIP预训练 + Swin-UNet + 世界模型 + RL训练总时间仍很长。

## 8. 与其他章节的联系

| 章节 | 联系 |
|------|------|
| 第7章 Transformer | MineCLIP 基于 Transformer |
| 第10章 自监督学习 | MineCLIP 是视频-文本对齐预训练 |
| 第15章 元学习 | 功用性图 = 学到的"如何探索"的元知识 |
| 第18章 XAI | 功用性图本身是黑盒模型的可解释性输出 |
| 第19章 ChatGPT | GPT(文本) + MineCLIP(视觉-文本) = 多模态 |

## 9. 核心公式

### Bootstrap λ-returns

$$R_t^\lambda = \hat{c}_t[\hat{G}_{t+1} + \gamma^{\hat{\Delta}_{t+1}}((1-\lambda)v_\psi(\hat{s}_{t+1})+\lambda R_{t+1}^\lambda)]$$

### 内在奖励

$$r_{\text{int}} = \frac{1}{HW}\sum_{i,j} M_{ij} \cdot G_{ij}$$

($M$=功用性图, $G$=二维高斯)

### 跳跃标志

$$\text{Jump} = \begin{cases}1 & \text{Kurtosis}(M) > \tau \\ 0 & \text{otherwise}\end{cases}$$

## 10. 关键总结

1. **解决"短视"问题**：长短期世界模型让智能体能看到未来
2. **功用性图是探索的"指南针"**：不依赖成功轨迹
3. **目标不可见时仍有效**：MineCLIP 学到的上下文知识
4. **跳跃式转换直接注入长期回报**：改进的 bootstrap λ-returns
5. **单卡3090纯视觉**：降低门槛，不开API
6. **所有任务显著优于 DreamerV3**
7. **功也图是信息枢纽**：探索指导 + 内在奖励 + 世界模型输入
8. **为开放世界RL提供新范式**：长期想象 + 功用性意识